# WaveProp — Hardware FPGA / PYNQ

Version matérielle WaveProp avec `waveprop_compute_0` et `axi_dma_0`.


In [ ]:
from pynq import Overlay, allocate
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time


## Chargement de l'overlay


In [ ]:
overlay = Overlay('design_1.bit')
print(overlay.ip_dict.keys())


## IP WaveProp et DMA


In [ ]:
dma = overlay.axi_dma_0
waveprop = overlay.waveprop_compute_0

print("DMA :", dma)
print("WaveProp :", waveprop)


## Buffer de sortie

WaveProp produit une image 100 × 100. Le buffer contient donc 10 000 `uint32`, soit 40 000 octets.


In [ ]:
SIZE = 100

output_buffer = allocate(shape=(SIZE * SIZE,), dtype=np.uint32)

print("Nombre de valeurs :", output_buffer.size)
print("Taille du buffer :", output_buffer.nbytes, "octets")


## Exécution matérielle

Le DMA S2MM est lancé avant l'IP WaveProp. Le flux est ensuite écrit en DDR via HP0.


In [ ]:
dma.recvchannel.transfer(output_buffer)

waveprop.write(0x00, 1)

dma.recvchannel.wait()
output_buffer.invalidate()

print("Transfert terminé")


## Mise en forme du résultat


In [ ]:
frame = np.array(output_buffer, dtype=np.uint32).reshape((SIZE, SIZE))

print("Min :", frame.min())
print("Max :", frame.max())


## Affichage


In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(frame, cmap="gray", vmin=0, vmax=255)
plt.title("WaveProp - FPGA")
plt.axis("off")
plt.colorbar()
plt.show()


## Animation de la propagation


In [ ]:
NB_FRAMES = 100

fig, ax = plt.subplots(figsize=(7, 7))

for i in range(NB_FRAMES):
    dma.recvchannel.transfer(output_buffer)

    waveprop.write(0x00, 1)

    dma.recvchannel.wait()
    output_buffer.invalidate()

    frame = np.array(output_buffer, dtype=np.uint32).reshape((SIZE, SIZE))

    ax.clear()
    ax.imshow(frame, cmap="gray", vmin=0, vmax=255)
    ax.set_title(f"WaveProp FPGA - Frame {i + 1}/{NB_FRAMES}")
    ax.axis("off")

    clear_output(wait=True)
    display(fig)
    time.sleep(0.05)

plt.close(fig)


## Libération du buffer


In [ ]:
output_buffer.freebuffer()
print("Buffer libéré")
